# N02 Main Evaluation Manifest Repair — v1.0.4 candidate

This notebook uses the uploaded **v1.0.3 audit ZIP** as an immutable source, repairs the confirmed weak/duplicated code-mixed examples, then **recomputes every validation after edits**. It does not rely on cached `checks` values from earlier cells.

Repairs target: 24 rows called out by the prior domain-cue report, the extra row from numeric-masked template duplicates, four previously reported syntax-error IDs, and the three long code examples specifically called out earlier. The target set is deduplicated before edits; source samples and threshold values are extracted from the source code wherever present.

**Important artifact mismatch:** in the exact v1.0.3 CSV included in the supplied ZIP, T039/T041/T048/T067 currently parse as valid Python; the pasted indentation errors apparently came from another intermediate candidate/runtime. This notebook still regenerates those four rows to add explicit domain context, robust missing/equal-value handling, and topic-specific function names. It will print the source syntax audit so the mismatch remains visible.

No activations are collected in this notebook. The new manifest remains a candidate until token, syntax, duplicate, topic-cue, structure checks pass and the edited rows receive human review.


In [ ]:
%pip -q install transformers pandas

In [ ]:
from pathlib import Path
import zipfile, json, hashlib, shutil, re, ast
import pandas as pd
import transformers
from IPython.display import display
from google.colab import files

print("Transformers:", transformers.__version__)
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Upload exactly one N02_Frozen_Manifest_Audit_v1_0_3.zip; got: {zip_names}")

SOURCE_ZIP = Path("/content") / zip_names[0]
WORK = Path("/content/n02_manifest_repair_v1_0_4")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP, "r") as zf:
    zf.extractall(WORK)

csvs = list(WORK.glob("*.csv"))
jsons = list(WORK.glob("*.json"))
if len(csvs) != 1 or len(jsons) != 1:
    raise ValueError(f"Expected one CSV and one JSON; got CSV={csvs}, JSON={jsons}")
SOURCE_MANIFEST, SOURCE_REPORT = csvs[0], jsons[0]
report = json.loads(SOURCE_REPORT.read_text(encoding="utf-8"))
source_bytes = SOURCE_MANIFEST.read_bytes()
source_file_hash = hashlib.sha256(source_bytes).hexdigest()
source_df = pd.read_csv(SOURCE_MANIFEST, keep_default_na=False)
df = source_df.copy(deep=True)

def sha256_bytes(blob):
    return hashlib.sha256(blob).hexdigest()

def text_fingerprint(frame):
    payload = "\n".join(
        f"{text_id}\n{text}"
        for text_id, text in zip(frame["text_id"].astype(str), frame["text"].astype(str))
    ).encode("utf-8")
    return sha256_bytes(payload)

if source_file_hash != report.get("manifest_file_sha256"):
    raise ValueError("Uploaded CSV hash does not match v1.0.3 report. Stop; do not repair an unverified source.")
if text_fingerprint(df) != report.get("candidate_text_manifest_sha256"):
    raise ValueError("Source text fingerprint does not match report. Stop and inspect the archive.")
if len(df) != 400 or df["topic_id"].nunique() != 100 or not df["text_id"].is_unique:
    raise ValueError("Unexpected source structure. Stop before repair.")

# Explicitly report syntax status of the exact CSV inside the ZIP.
source_code = df.loc[df["style"] == "code_plus_natural_language"]
def syntax_error(text):
    try:
        ast.parse(str(text))
        return ""
    except SyntaxError as exc:
        return f"{type(exc).__name__}: {exc.msg} (line {exc.lineno})"
source_code = source_code.copy()
source_code["source_syntax_error"] = source_code["text"].map(syntax_error)
source_errors = source_code.loc[source_code["source_syntax_error"] != "", ["topic_id", "text_id", "source_syntax_error"]]
print("Verified source file SHA-256:", source_file_hash)
print("Verified source text fingerprint:", text_fingerprint(df))
print("Code syntax errors in exact uploaded CSV:", len(source_errors))
if len(source_errors):
    display(source_errors)
else:
    print("The four pasted IndentationErrors were not present in this exact ZIP copy; those IDs will still be regenerated below.")


## 1. Select repair targets and regenerate topic-specific code

Code rows are regenerated from the existing `topic_domain`, `operation_id`, original numeric sample list, original threshold (where needed), and length tier. This reduces reliance on manually guessed numbers and produces meaningful domain context and identifiers. All changes remain drafts pending review.


In [ ]:
DOMAIN_INFO = {
    "greenhouse_temperature": {
        "context": "greenhouse probe temperature readings",
        "short": "greenhouse temperature",
        "unit": "deg C",
        "variable": "greenhouse_temperature",
        "noun": "greenhouse temperature",
        "cues": ["greenhouse", "temperature", "probe"],
    },
    "solar_output": {
        "context": "solar inverter power-output readings",
        "short": "solar inverter output",
        "unit": "kW",
        "variable": "solar_output",
        "noun": "solar power output",
        "cues": ["solar", "inverter", "power output"],
    },
    "coolant_pressure": {
        "context": "coolant-line pressure readings",
        "short": "coolant pressure",
        "unit": "kPa",
        "variable": "coolant_pressure",
        "noun": "coolant pressure",
        "cues": ["coolant", "pressure"],
    },
    "pump_flow": {
        "context": "pump-sensor flow-rate readings",
        "short": "pump flow rate",
        "unit": "L/min",
        "variable": "pump_flow",
        "noun": "pump flow rate",
        "cues": ["pump", "flow", "flow-rate"],
    },
    "cell_voltage": {
        "context": "battery cell-voltage readings",
        "short": "battery cell voltage",
        "unit": "V",
        "variable": "cell_voltage",
        "noun": "battery cell voltage",
        "cues": ["battery", "cell voltage", "voltage"],
    },
    "air_particles": {
        "context": "PM2.5 particle concentrations from air-quality monitors",
        "short": "air-quality particle counts",
        "unit": "ug/m3",
        "variable": "air_particles",
        "noun": "air-quality particle concentration",
        "cues": ["air-quality", "particle", "pm2.5"],
    },
    "engine_fuel_rate": {
        "context": "engine test-bench fuel-rate readings",
        "short": "engine fuel rate",
        "unit": "L/h",
        "variable": "engine_fuel_rate",
        "noun": "engine fuel rate",
        "cues": ["engine", "fuel-rate", "fuel rate"],
    },
    "network_delay": {
        "context": "network packet-delay readings from trace logs",
        "short": "network packet delay",
        "unit": "ms",
        "variable": "network_delay",
        "noun": "network packet delay",
        "cues": ["network", "packet-delay", "packet delay"],
    },
    "service_latency": {
        "context": "service response-latency readings from health checks",
        "short": "service response latency",
        "unit": "ms",
        "variable": "service_latency",
        "noun": "service response latency",
        "cues": ["service", "latency", "response"],
    },
    "turbine_vibration": {
        "context": "turbine-sensor vibration amplitudes",
        "short": "turbine vibration",
        "unit": "mm/s",
        "variable": "turbine_vibration",
        "noun": "turbine vibration amplitude",
        "cues": ["turbine", "vibration"],
    },
}

OP_TASK = {
    "count_threshold": "count observations at or above a stated threshold",
    "deduplicate": "remove repeated readings while preserving the first occurrence",
    "filter_threshold": "retain readings strictly above the stated threshold",
    "max_index": "return the index and value of the maximum reading",
    "mean": "compute the arithmetic mean of the readings",
    "median": "compute the median of the readings",
    "minmax_normalize": "map the observations to the zero-to-one range using min-max normalization",
    "moving_average": "compute a three-sample moving average",
    "range": "calculate the maximum minus the minimum",
    "sort_ascending": "sort the readings from smallest to largest",
}

# Targets = 24 domain-cue warning IDs + the remaining row from duplicate-template groups
# + four earlier syntax-error IDs + two additional long rows reviewed previously.
TARGET_IDS = sorted(set([
    "T002", "T007", "T014", "T024", "T032", "T034", "T035", "T037",
    "T038", "T040", "T042", "T043", "T044", "T046", "T050", "T055",
    "T056", "T064", "T065", "T066", "T088", "T093", "T094", "T096",
    "T061", "T039", "T041", "T048", "T067", "T085", "T090"
]))

assert len(TARGET_IDS) == 31, f"Unexpected repair target count: {len(TARGET_IDS)}"
code_mask = df["style"] == "code_plus_natural_language"
missing_target = set(TARGET_IDS) - set(df.loc[code_mask, "topic_id"].astype(str))
if missing_target:
    raise ValueError(f"Target topics missing from source code rows: {sorted(missing_target)}")

def assigned_name(node):
    names = []
    for target in node.targets:
        if isinstance(target, ast.Name):
            names.append(target.id)
    return names

def extract_sample_values(source):
    tree = ast.parse(str(source))
    candidates = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and isinstance(node.value, ast.List):
            vals = []
            okay = True
            for element in node.value.elts:
                if not (isinstance(element, ast.Constant) and isinstance(element.value, (int, float)) and not isinstance(element.value, bool)):
                    okay = False
                    break
                vals.append(element.value)
            if okay and vals:
                names = assigned_name(node)
                preference = 2 if any(n in {"sample", "values", "readings", "x"} for n in names) else 1
                candidates.append((preference, getattr(node, "lineno", 0), vals))
    if not candidates:
        raise ValueError("Could not extract original numeric sample; refusing to invent values.")
    candidates.sort(key=lambda item: (item[0], item[1]))
    return candidates[-1][2]

def extract_threshold(source):
    tree = ast.parse(str(source))
    # Prefer a default value on an argument named threshold.
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            args = [a.arg for a in node.args.args]
            defaults = node.args.defaults
            start = len(args) - len(defaults)
            for offset, default in enumerate(defaults):
                arg_name = args[start + offset]
                if arg_name == "threshold" and isinstance(default, ast.Constant) and isinstance(default.value, (int, float)):
                    return default.value
    # Otherwise use an explicit threshold keyword argument.
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            for kw in node.keywords:
                if kw.arg == "threshold" and isinstance(kw.value, ast.Constant) and isinstance(kw.value.value, (int, float)):
                    return kw.value.value
    # Or a direct local assignment.
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and "threshold" in assigned_name(node):
            if isinstance(node.value, ast.Constant) and isinstance(node.value.value, (int, float)):
                return node.value.value
    raise ValueError("Threshold operation has no recoverable original threshold; refusing to invent one.")

def fmt_num(v):
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return repr(v)

def threshold_value(row, old_text):
    if row["operation_id"] in {"count_threshold", "filter_threshold"}:
        return extract_threshold(old_text)
    return None

def expr_for(operation, q=None):
    exprs = {
        "deduplicate": "list(dict.fromkeys(values))",
        "filter_threshold": f"[v for v in values if v > {fmt_num(q)}]",
        "count_threshold": f"sum(v >= {fmt_num(q)} for v in values)",
        "max_index": "max(enumerate(values), key=lambda pair: pair[1]) if values else None",
        "mean": "sum(values) / len(values) if values else None",
        "median": "median(values) if values else None",
        "minmax_normalize": "[(v - low) / (high - low) for v in values]",
        "moving_average": "[sum(values[i:i+3]) / 3 for i in range(len(values) - 2)]",
        "range": "max(values) - min(values) if values else None",
        "sort_ascending": "sorted(values)",
    }
    return exprs[operation]

def function_body(operation, q=None):
    lines = ["    if not values:", "        return None", "    if any(v is None for v in values):", "        raise ValueError('Missing measurement value')"]
    if operation == "minmax_normalize":
        lines += [
            "    low, high = min(values), max(values)",
            "    if low == high:",
            "        return [0.0 for _ in values]",
            "    return [(v - low) / (high - low) for v in values]",
        ]
    elif operation == "moving_average":
        lines += [
            "    if len(values) < 3:",
            "        return []",
            "    return [sum(values[i:i+3]) / 3 for i in range(len(values) - 2)]",
        ]
    else:
        lines.append("    return " + expr_for(operation, q))
    return "\n".join(lines)

def short_code(info, operation, nums, q):
    arr = ", ".join(fmt_num(v) for v in nums)
    lines = [f"# {info['short']} ({info['unit']}).", f"x = [{arr}]"]
    if operation in {"count_threshold", "filter_threshold"}:
        lines.append(f"q = {fmt_num(q)}")
    if operation == "minmax_normalize":
        lines += [
            "a, b = min(x), max(x)",
            "y = [(v-a)/(b-a) for v in x] if b > a else [0.0]*len(x)",
        ]
    else:
        short_expr = {
            "deduplicate": "list(dict.fromkeys(x))",
            "filter_threshold": "[v for v in x if v > q]",
            "count_threshold": "sum(v >= q for v in x)",
            "max_index": "max(enumerate(x), key=lambda p: p[1]) if x else None",
            "mean": "sum(x)/len(x) if x else None",
            "median": "median(x) if x else None",
            "moving_average": "[sum(x[i:i+3])/3 for i in range(len(x)-2)]",
            "range": "max(x)-min(x) if x else None",
            "sort_ascending": "sorted(x)",
        }
        if operation == "median":
            lines.insert(1, "from statistics import median")
        lines.append("y = " + short_expr[operation])
    lines.append("print(y)")
    return "\n".join(lines)

def medium_code(info, operation, nums, q, topic_id):
    arr = ", ".join(fmt_num(v) for v in nums)
    func = f"{operation}_{info['variable']}"
    param = f", threshold={fmt_num(q)}" if operation in {"count_threshold", "filter_threshold"} else ""
    call = f"{func}(sample, threshold={fmt_num(q)})" if operation in {"count_threshold", "filter_threshold"} else f"{func}(sample)"
    pre = "from statistics import median\n" if operation == "median" else ""
    lines = [
        f"# Context: {info['context']} ({info['unit']}).",
        f"def {func}(values{param}):",
        *function_body(operation, q).split("\n"),
        "",
        f"sample = [{arr}]",
        f"result = {call}",
        f"print('{info['short']}:', result)",
        f"# Task: {OP_TASK[operation]}",
    ]
    return pre + "\n".join(lines)

def long_code(info, operation, nums, q, topic_id):
    arr = ", ".join(fmt_num(v) for v in nums)
    func = f"{operation}_{info['variable']}"
    param = f", threshold={fmt_num(q)}" if operation in {"count_threshold", "filter_threshold"} else ""
    call = f"{func}(observations, threshold={fmt_num(q)})" if operation in {"count_threshold", "filter_threshold"} else f"{func}(observations)"
    pre = "from statistics import median\n" if operation == "median" else ""
    lines = [
        f"# Context: {info['context']} ({info['unit']}) from one observation window.",
        f"def {func}(values{param}):",
        f"    # {OP_TASK[operation].capitalize()}.",
        *function_body(operation, q).split("\n"),
        "",
        f"observations = [{arr}]",
        f"result = {call}",
        f"print('Input ({info['unit']}):', observations)",
        f"print('Result:', result)",
        f"# Task: {OP_TASK[operation].capitalize()} for {info['context']}.",
        "# Interpretation: the output summarizes only the supplied observation window.",
        "# Validation: document missing-value handling, threshold choice, and units before comparing runs.",
        "# Limitation: this numerical operation alone does not establish sensor calibration or diagnose the physical system.",
    ]
    return pre + "\n".join(lines)

def make_replacement(row):
    domain = str(row["topic_domain"])
    operation = str(row["operation_id"])
    tier = str(row["length_tier"])
    if domain not in DOMAIN_INFO:
        raise ValueError(f"Unknown topic domain: {domain}")
    if operation not in OP_TASK:
        raise ValueError(f"Unknown operation: {operation}")
    old = str(row["text"])
    nums = extract_sample_values(old)
    q = threshold_value(row, old)
    info = DOMAIN_INFO[domain]
    if tier == "short":
        return short_code(info, operation, nums, q)
    if tier == "medium":
        return medium_code(info, operation, nums, q, str(row["topic_id"]))
    if tier == "long":
        return long_code(info, operation, nums, q, str(row["topic_id"]))
    raise ValueError(f"Unknown length tier: {tier}")

# Capture original content for a transparent change log.
change_rows = []
for topic_id in TARGET_IDS:
    mask = (df["topic_id"].astype(str) == topic_id) & (df["style"] == "code_plus_natural_language")
    if int(mask.sum()) != 1:
        raise ValueError(f"Expected one code row for {topic_id}; found {int(mask.sum())}")
    idx = df.index[mask][0]
    old_text = str(df.at[idx, "text"])
    replacement = make_replacement(df.loc[idx])
    df.at[idx, "text"] = replacement
    change_rows.append({
        "topic_id": topic_id,
        "topic_domain": str(df.at[idx, "topic_domain"]),
        "operation_id": str(df.at[idx, "operation_id"]),
        "length_tier": str(df.at[idx, "length_tier"]),
        "text_id": str(df.at[idx, "text_id"]),
        "old_text": old_text,
        "new_text": replacement,
    })

changes_df = pd.DataFrame(change_rows)
print("Rows regenerated:", len(changes_df))
display(changes_df[["topic_id", "topic_domain", "operation_id", "length_tier", "text_id", "new_text"]].sort_values("topic_id"))


## 2. Recompute every check from the edited manifest

All checks below use the current `df` **after** applying the replacements. Token counts are remeasured for all 400 rows using both tokenizers, rather than reusing the original count columns. Any failing row blocks freeze.


In [ ]:
tokenizer_specs = {
    "gpt2": "gpt2",
    "pythia-160m": "EleutherAI/pythia-160m",
}
tokenizers = {key: transformers.AutoTokenizer.from_pretrained(model_id) for key, model_id in tokenizer_specs.items()}
bands = {"short": (16, 64), "medium": (65, 128), "long": (129, 256)}

for key, tokenizer in tokenizers.items():
    token_col, band_col = f"tokens_{key}", f"in_band_{key}"
    df[token_col] = [len(tokenizer.encode(str(text), add_special_tokens=True)) for text in df["text"]]
    df[band_col] = [
        bands[str(tier)][0] <= int(count) <= bands[str(tier)][1]
        for tier, count in zip(df["length_tier"], df[token_col])
    ]

# Syntax check of all code-mixed rows.
code = df.loc[df["style"] == "code_plus_natural_language"].copy()
code["syntax_error"] = code["text"].map(syntax_error)

# Topic cue check with aliases, not raw snake_case only.
def has_domain_cue(row):
    info = DOMAIN_INFO.get(str(row["topic_domain"]))
    if info is None:
        return False
    text = str(row["text"]).lower()
    return any(cue.lower() in text for cue in info["cues"])
code["topic_cue_present"] = code.apply(has_domain_cue, axis=1)

# Normalize whitespace and replace numbers only. Keep comments and identifiers so a
# topic-specific sample/function does not become a false duplicate merely because
# two rows implement the same operation.
def numeric_masked_signature(text):
    text = re.sub(r"\b\d+(?:\.\d+)?\b", "<NUM>", str(text).lower())
    return re.sub(r"\s+", " ", text).strip()
code["numeric_masked_signature"] = code["text"].map(numeric_masked_signature)
code["exact_duplicate"] = code["text"].duplicated(keep=False)
code["numeric_masked_duplicate"] = code["numeric_masked_signature"].duplicated(keep=False)
code["generic_docstring"] = code["text"].str.contains(
    r"Task execution docstring\.",
    case=False, regex=True
)

# Full dataset structure is recomputed after replacements.
structure_checks = {
    "rows_400": len(df) == 400,
    "topics_100": df["topic_id"].nunique() == 100,
    "unique_text_id": bool(df["text_id"].is_unique),
    "one_row_per_topic_style": int(df[["topic_id", "style"]].duplicated().sum()) == 0,
    "four_styles_per_topic": bool((df.groupby("topic_id")["style"].nunique() == 4).all()),
    "one_length_tier_per_topic": bool((df.groupby("topic_id")["length_tier"].nunique() == 1).all()),
    "nonempty_text": bool(df["text"].astype(str).str.strip().ne("").all()),
    "no_exact_duplicates_across_dataset": int(df["text"].duplicated().sum()) == 0,
    "pilot_prompts_excluded": bool(df["pilot_excluded"].astype(str).str.lower().isin(["true"]).all()),
}

out_of_band = df.loc[~(df["in_band_gpt2"] & df["in_band_pythia-160m"]), [
    "topic_id", "topic_domain", "length_tier", "style", "text_id",
    "tokens_gpt2", "in_band_gpt2", "tokens_pythia-160m", "in_band_pythia-160m", "text"
]].copy()
syntax_failures = code.loc[code["syntax_error"] != "", ["topic_id", "text_id", "syntax_error", "text"]].copy()
domain_failures = code.loc[~code["topic_cue_present"], ["topic_id", "topic_domain", "text_id", "text"]].copy()
exact_duplicates = code.loc[code["exact_duplicate"], ["topic_id", "topic_domain", "text_id", "text"]].copy()
masked_duplicates = code.loc[code["numeric_masked_duplicate"], ["topic_id", "topic_domain", "operation_id", "text_id", "text"]].copy()
generic_docs = code.loc[code["generic_docstring"], ["topic_id", "topic_domain", "text_id", "text"]].copy()

check_summary = {
    "n_rows": int(len(df)),
    "n_topics": int(df["topic_id"].nunique()),
    "n_code_rows": int(len(code)),
    "structural_checks": structure_checks,
    "tokenizer_out_of_band_rows": int(len(out_of_band)),
    "syntax_failure_rows": int(len(syntax_failures)),
    "domain_cue_missing_rows": int(len(domain_failures)),
    "exact_duplicate_code_rows": int(len(exact_duplicates)),
    "numeric_masked_template_duplicate_rows": int(len(masked_duplicates)),
    "generic_docstring_rows": int(len(generic_docs)),
}
check_summary["hard_checks_pass"] = all(structure_checks.values()) and all([
    len(out_of_band) == 0,
    len(syntax_failures) == 0,
    len(domain_failures) == 0,
    len(exact_duplicates) == 0,
    len(masked_duplicates) == 0,
    len(generic_docs) == 0,
])

print("RECOMPUTED CHECK SUMMARY")
display(pd.DataFrame([{"check": k, "result": json.dumps(v, ensure_ascii=False) if isinstance(v, dict) else v} for k, v in check_summary.items()]))
print("\nTokenizer summary")
display(pd.DataFrame([
    {"tokenizer": key, "out_of_band_rows": int((~df[f"in_band_{key}"]).sum()),
     "min_tokens": int(df[f"tokens_{key}"].min()), "max_tokens": int(df[f"tokens_{key}"].max())}
    for key in tokenizer_specs
]))


## 3. Inspect failures and changed rows before approval

The following tables are the fresh validation results after edits. If any hard-check failure remains, fix those rows and rerun the relevant edit plus the full validation cell. Do not reuse old warnings or checks.


In [ ]:
for label, table in [
    ("Syntax failures", syntax_failures),
    ("Topic cue missing", domain_failures),
    ("Exact duplicate code", exact_duplicates),
    ("Numeric-masked template duplicates", masked_duplicates),
    ("Generic placeholder docstrings", generic_docs),
    ("Out-of-band rows", out_of_band),
]:
    print(f"\n=== {label}: {len(table)} row(s) ===")
    if len(table):
        display(table)

print("\n=== All regenerated code rows (human review required):", len(changes_df), "===")
display(changes_df[["topic_id", "topic_domain", "operation_id", "length_tier", "text_id", "new_text"]].sort_values("topic_id"))

# Save a review-only package even if hard checks do not pass; this is NOT a freeze.
REVIEW_DIR = Path("/content/n02_manifest_repair_v1_0_4_review")
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(REVIEW_DIR / "N02_Main_Evaluation_CANDIDATE_Manifest_v1.0.4.csv", index=False)
changes_df.to_csv(REVIEW_DIR / "N02_v1.0.4_code_text_change_log.csv", index=False)
with open(REVIEW_DIR / "N02_v1.0.4_recomputed_checks.json", "w", encoding="utf-8") as f:
    json.dump(check_summary, f, ensure_ascii=False, indent=2)
print("Review artifacts:", REVIEW_DIR)


## 4. Freeze gate — intentionally fails closed

Inspect all regenerated code examples, their matching non-code styles, the token-band table, and every failure table. Only then set `HUMAN_REVIEW_APPROVED = True`. The cell re-runs the entire validation immediately before freezing, so it cannot reuse a stale check result.


In [ ]:
HUMAN_REVIEW_APPROVED = False  # Set True only after reviewing all changed rows and matching styles.

# Recompute all checks one last time from the current df; do not trust stale checks.
# Tokenizer counts / validation can be expensive, but this final pass is deliberate.
final_df = df.copy(deep=True)
for key, tokenizer in tokenizers.items():
    final_df[f"tokens_{key}"] = [len(tokenizer.encode(str(t), add_special_tokens=True)) for t in final_df["text"]]
    final_df[f"in_band_{key}"] = [
        bands[str(tier)][0] <= int(count) <= bands[str(tier)][1]
        for tier, count in zip(final_df["length_tier"], final_df[f"tokens_{key}"])
    ]

final_code = final_df.loc[final_df["style"] == "code_plus_natural_language"].copy()
final_code["syntax_error"] = final_code["text"].map(syntax_error)
final_code["topic_cue_present"] = final_code.apply(has_domain_cue, axis=1)
final_code["numeric_masked_signature"] = final_code["text"].map(numeric_masked_signature)
final_code["generic_docstring"] = final_code["text"].str.contains(
    r"Task execution docstring\.",
    case=False, regex=True
)
final_structural = {
    "rows_400": len(final_df) == 400,
    "topics_100": final_df["topic_id"].nunique() == 100,
    "unique_text_id": bool(final_df["text_id"].is_unique),
    "one_row_per_topic_style": int(final_df[["topic_id", "style"]].duplicated().sum()) == 0,
    "four_styles_per_topic": bool((final_df.groupby("topic_id")["style"].nunique() == 4).all()),
    "one_length_tier_per_topic": bool((final_df.groupby("topic_id")["length_tier"].nunique() == 1).all()),
    "nonempty_text": bool(final_df["text"].astype(str).str.strip().ne("").all()),
    "no_exact_duplicates_across_dataset": int(final_df["text"].duplicated().sum()) == 0,
    "pilot_prompts_excluded": bool(final_df["pilot_excluded"].astype(str).str.lower().isin(["true"]).all()),
}
final_token_pass = bool((final_df["in_band_gpt2"] & final_df["in_band_pythia-160m"]).all())
final_hard_pass = all(final_structural.values()) and final_token_pass and all([
    bool(final_code["syntax_error"].eq("").all()),
    bool(final_code["topic_cue_present"].all()),
    int(final_code["text"].duplicated().sum()) == 0,
    int(final_code["numeric_masked_signature"].duplicated().sum()) == 0,
    bool((~final_code["generic_docstring"]).all()),
])

print("FINAL HARD CHECK PASS:", final_hard_pass)
print("HUMAN REVIEW APPROVED:", HUMAN_REVIEW_APPROVED)

if not final_hard_pass:
    print("NOT FROZEN: one or more hard checks failed. Review failure tables, edit, then rerun validations.")
elif not HUMAN_REVIEW_APPROVED:
    print("NOT FROZEN: hard checks passed, but manual review is not approved.")
else:
    final_df["role"] = "main_eval_frozen"
    frozen_csv = Path("/content/N02_Main_Evaluation_FROZEN_Manifest_v1.0.4.csv")
    final_df.to_csv(frozen_csv, index=False, lineterminator="\n")

    text_hash = text_fingerprint(final_df)
    file_hash = sha256_bytes(frozen_csv.read_bytes())
    report_out = {
        "protocol_version": "N02_Main_Evaluation_Dataset_Design_v1.0.4",
        "manifest_revision": "v1.0.4",
        "source_manifest_revision": "v1.0.3_candidate",
        "source_manifest_file_sha256": source_file_hash,
        "frozen_manifest_file_sha256": file_hash,
        "candidate_text_manifest_sha256": text_hash,
        "rows": int(len(final_df)),
        "topic_clusters": int(final_df["topic_id"].nunique()),
        "styles_per_topic": 4,
        "tokenizers": tokenizer_specs,
        "length_bands": bands,
        "hard_check_failures": [],
        "all_candidates_in_band_for_both_tokenizers": True,
        "human_content_review_complete": True,
        "frozen_for_confirmatory_analysis": True,
        "replaced_text_ids": [r["text_id"] for r in change_rows],
        "replacement_count": int(len(change_rows)),
        "resampling_unit_for_matched_style_contrasts": "topic_id cluster; retain all four styles per resampled topic",
        "bootstrap_replicates": 10000,
        "pilot_prompts_excluded": True,
        "role_in_manifest": ["main_eval_frozen"],
        "validation_recomputed_after_edits": True,
        "validation_recomputed_at_freeze": True,
        "code_syntax_failures": 0,
        "code_topic_cue_missing_rows": 0,
        "numeric_masked_template_duplicate_rows": 0,
        "generic_docstring_rows": 0,
        "notes": "v1.0.4 created as a new revision from immutable v1.0.3 candidate source. Original numeric sample values and operation thresholds were extracted from source code for edited rows. No model activations were captured in this notebook.",
    }
    report_path = Path("/content/N02_preflight_report_v1.0.4.json")
    report_path.write_text(json.dumps(report_out, ensure_ascii=False, indent=2), encoding="utf-8")
    zip_path = Path("/content/N02_Frozen_Manifest_Audit_v1_0_4.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(frozen_csv, frozen_csv.name)
        z.write(report_path, report_path.name)
        z.write(REVIEW_DIR / "N02_v1.0.4_code_text_change_log.csv", "N02_v1.0.4_code_text_change_log.csv")
    print("FROZEN MANIFEST:", frozen_csv)
    print("Manifest SHA-256:", file_hash)
    print("Text fingerprint:", text_hash)
    print("Audit ZIP:", zip_path)
    files.download(str(zip_path))


## Acceptance checklist

- [ ] Original v1.0.3 CSV/report hashes verified and preserved.
- [ ] The 31 target code rows regenerated from existing sample values and thresholds.
- [ ] Every edited code row reviewed against its general-prose, question/dialogue, and academic topic variants.
- [ ] Syntax and domain-cue checks recomputed across all 100 code rows after edits.
- [ ] Exact duplicate and number-masked template checks recomputed after edits.
- [ ] Token counts recomputed for all 400 rows with GPT-2 and Pythia tokenizers.
- [ ] All structural and token-band checks pass.
- [ ] `HUMAN_REVIEW_APPROVED` set to True only after actual review.
- [ ] Only the new v1.0.4 manifest is used for confirmatory activation capture after a successful freeze.
